##### Citation Info
@InProceedings{Gushchin_2026_CVPR,
    author    = {Gushchin, Aleksandr and Abud, Khaled and Shumitskaya, Ekaterina and Filippov, Artem and Bychkov, Georgii and Lavrushkin, Sergey and Erofeev, Mikhail and Antsiferova, Anastasia and Chen, Changsheng and Tan, Shunquan and Timofte, Radu and Vatolin, Dmitriy and Song, Chuanbiao and Yu, Zijian and Tan, Hao and Lan, Jun and Yang, Zhiqiang and Tang, Yongwei and Wu, Zhiqiang and Seow, Jia Wen and Koay, Hong Vin and Ren, Haodong and Xu, Feng and Chen, Shuai and Xia, Ruiyang and Zhang, Qi and Xu, Yaowen and Zou, Zhaofan and Sun, Hao and Lu, Dagong and Yao, Mufeng and Xu, Xinlei and Wu, Fei and Guo, Fengjun and Luo, Cong and Sharma, Hardik and Negi, Aashish and Shaily, Prateek and Kumar, Jayant and Chaudhary, Sachin and Dudhane, Akshay and Hambarde, Praful and Shukla, Amit and Tu, Zhilin and Li, Fengpeng and Zhang, Jiamin and Fei, Jianwei and Li, Kemou and Wu, Haiwei and Benjdira, Bilel and Ali, Anas M. and Boulila, Wadii and Qu, Chenfan and Li, Junchi},
    title     = {NTIRE 2026 Challenge on Robust AI-Generated Image Detection in the Wild},
    booktitle = {Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR) Workshops},
    month     = {June},
    year      = {2026},
    pages     = {1895-1913}
}


In [ ]:
import torch
import torchvision

# check NVIDIA GPU available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Set up image transforms, datasets, and data loaders

from torch.utils.data import DataLoader
from torchvision.models import ResNet50_Weights
from torchvision import transforms
from official_augmentations import OfficialDistortions
from aigendet_dataset import AIGenDetDataset

# Evaluation remains deterministic and uses the pretrained weights' preprocessing.
transform = ResNet50_Weights.DEFAULT.transforms()

# Apply official distortions 50% of the time with max of 3 distortions and 5 levels.
official_distortions = OfficialDistortions(max_distortions=3, num_levels=5)
train_transform = transforms.Compose([
    transforms.Resize(
        transform.resize_size, interpolation=transform.interpolation,
        antialias=transform.antialias,
    ),
    transforms.CenterCrop(transform.crop_size),
    transforms.ToTensor(),
    transforms.RandomApply([official_distortions], p=0.50),
    transforms.Normalize(mean=transform.mean, std=transform.std),
])

batch_size = 32

train_path = "../data/train"
train_shard_nums = [0, 5] # list of training shards to use (0-5)
train = AIGenDetDataset(train_path, train_shard_nums, transform=train_transform)
val = AIGenDetDataset(
    image_dir="../data/val/val/val_images",
    labels_csv="../data/val/val/val_labels.csv",
    transform=transform,
)
val_hard = AIGenDetDataset(
    image_dir="../data/val/val_hard/val_images_hard",
    labels_csv="../data/val/val_hard/val_hard_labels.csv",
    transform=transform,
)
test = AIGenDetDataset(
    image_dir="../data/test/test_images",
    labels_csv="../data/test/test_labels.csv",
    transform=transform,
)

trainloader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
valloader = DataLoader(val, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)
val_hardloader = DataLoader(val_hard, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

# Creating this loader does not run test evaluation.
testloader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)


In [ ]:
# Set up model for fine tuning

import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights.DEFAULT)

# Replace the final layer with a new one that has 2 output classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

for name, parameter in model.named_parameters():
    parameter.requires_grad = name.startswith(
        ("layer3.", "layer4.", "fc.")
    )

model.to(device)

In [ ]:
# set up optimizer

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-4},
])

In [ ]:
checkpoint = torch.load(
    "../models/ResNet50_official_resume.pth",
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

start_epoch = checkpoint["epoch"]  # Repeat the interrupted epoch.
best_roc_auc = checkpoint["best_roc_auc"]
miss_count = checkpoint["miss_count"]

In [ ]:
# Train/eval loop
from sklearn.metrics import roc_auc_score

best_roc_auc = float('-inf')
miss_count = 0
max_misses = 5

for epoch in range(start_epoch, 50):
    model.train()

    running_loss = 0.0

    for i, batch in enumerate(trainloader, 0):
        images = batch["image"].to(device)
        binary_labels = batch["label"].to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, binary_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 250 == 249:    # print every 250 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 250:.6f}')
            running_loss = 0.0
    print(f'Finished epoch {epoch + 1}')
    
    # Validation
    model.eval()
    y_true, y_ai_probability = [], []

    with torch.no_grad():
        for batch in valloader:
            images = batch["image"].to(device)
            ai_probability = model(images).softmax(dim=1)[:, 1]

            y_true.extend(batch["label"].tolist())
            y_ai_probability.extend(ai_probability.cpu().tolist())

    val_roc_auc = roc_auc_score(y_true, y_ai_probability)
    print(f"Validation ROC AUC: {val_roc_auc:.6f}")

    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        miss_count = 0
        torch.save(model, "../models/ResNet50FineTuned_official_distortions.pth")
    else:
        miss_count += 1
        if miss_count >= max_misses:
            print("ROC AUC stopped improving, stopping early.")
            break

print('Finished Training')

In [ ]:
# Get detailed metrics for the current model on the validation set
model = torch.load('../models/ResNet50FineTuned_official_distortions.pth', map_location=device, weights_only=False)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

y_true, y_pred, y_ai_probability, is_distorted = [], [], [], []

model.eval()
with torch.no_grad():
    for batch in valloader:
        images = batch["image"].to(device)
        outputs = model(images)
        probabilities = outputs.softmax(dim=1)
        predictions = probabilities.argmax(dim=1)

        y_true.extend(batch["label"].to(device).tolist())
        y_pred.extend(predictions.tolist())
        # Class 1 is AI-generated; use its probability for every image.
        y_ai_probability.extend(probabilities[:, 1].tolist())
        is_distorted.extend(batch["is_distorted"].tolist())

# Keep the labels, predictions, scores, and distortion flags aligned.
y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)
y_ai_probability = np.asarray(y_ai_probability)
is_distorted = np.asarray(is_distorted)

# Classification uses the existing argmax decision (equivalent to threshold 0.5,
# with ties assigned to Real). ROC AUC uses continuous AI probabilities.
for subset_name, mask in [
    ("Overall", np.ones(len(y_true), dtype=bool)),
    ("Clean", is_distorted == 0),
    ("Distorted (robustness)", is_distorted == 1),
]:
    print(f"\n{subset_name}: {mask.sum()} images")
    if not mask.any():
        print("No images in this subset; skipping metrics.")
        continue

    labels = y_true[mask]
    predicted = y_pred[mask]
    scores = y_ai_probability[mask]
    print(classification_report(
        labels,
        predicted,
        labels=[0, 1],
        target_names=["Real", "AI-generated"],
        digits=3,
        zero_division=0,
    ))
    print("Confusion matrix (rows: actual, columns: predicted; order: Real, AI-generated):")
    print(confusion_matrix(labels, predicted, labels=[0, 1]))
    print(f"Balanced accuracy: {balanced_accuracy_score(labels, predicted):.4f}")
    print(f"Brier score (lower is better): {brier_score_loss(labels, scores):.4f}")
    print(f"Log loss (lower is better): {log_loss(labels, scores, labels=[0, 1]):.4f}")
    if len(np.unique(labels)) == 2:
        print(f"ROC-AUC (higher is better): {roc_auc_score(labels, scores):.6f}")
    else:
        print("ROC-AUC is undefined: this subset contains only one class.")

# Near the diagonal means predicted AI probabilities match observed AI frequencies.
CalibrationDisplay.from_predictions(
    y_true,
    y_ai_probability,
    n_bins=10,
    strategy="uniform",
    name="ResNet50 tuned",
)
plt.title("Validation calibration: AI-generated probability")
plt.tight_layout()
plt.show()


In [ ]:
# Get detailed metrics for the current model on the hard validation set
model = torch.load('../models/ResNet50FineTuned_official_distortions.pth', map_location=device, weights_only=False)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

y_true, y_pred, y_ai_probability, is_distorted = [], [], [], []

model.eval()
with torch.no_grad():
    for batch in val_hardloader:
        images = batch["image"].to(device)
        outputs = model(images)
        probabilities = outputs.softmax(dim=1)
        predictions = probabilities.argmax(dim=1)

        y_true.extend(batch["label"].to(device).tolist())
        y_pred.extend(predictions.tolist())
        # Class 1 is AI-generated; use its probability for every image.
        y_ai_probability.extend(probabilities[:, 1].tolist())
        is_distorted.extend(batch["is_distorted"].tolist())

# Keep the labels, predictions, scores, and distortion flags aligned.
y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)
y_ai_probability = np.asarray(y_ai_probability)
is_distorted = np.asarray(is_distorted)

# Classification uses the existing argmax decision (equivalent to threshold 0.5,
# with ties assigned to Real). ROC AUC uses continuous AI probabilities.
for subset_name, mask in [
    ("Overall", np.ones(len(y_true), dtype=bool)),
    ("Clean", is_distorted == 0),
    ("Distorted (robustness)", is_distorted == 1),
]:
    print(f"\n{subset_name}: {mask.sum()} images")
    if not mask.any():
        print("No images in this subset; skipping metrics.")
        continue

    labels = y_true[mask]
    predicted = y_pred[mask]
    scores = y_ai_probability[mask]
    print(classification_report(
        labels,
        predicted,
        labels=[0, 1],
        target_names=["Real", "AI-generated"],
        digits=3,
        zero_division=0,
    ))
    print("Confusion matrix (rows: actual, columns: predicted; order: Real, AI-generated):")
    print(confusion_matrix(labels, predicted, labels=[0, 1]))
    print(f"Balanced accuracy: {balanced_accuracy_score(labels, predicted):.4f}")
    print(f"Brier score (lower is better): {brier_score_loss(labels, scores):.4f}")
    print(f"Log loss (lower is better): {log_loss(labels, scores, labels=[0, 1]):.4f}")
    if len(np.unique(labels)) == 2:
        print(f"ROC-AUC (higher is better): {roc_auc_score(labels, scores):.6f}")
    else:
        print("ROC-AUC is undefined: this subset contains only one class.")

# Near the diagonal means predicted AI probabilities match observed AI frequencies.
CalibrationDisplay.from_predictions(
    y_true,
    y_ai_probability,
    n_bins=10,
    strategy="uniform",
    name="ResNet50 tuned (hard validation)",
)
plt.title("Validation calibration: AI-generated probability")
plt.tight_layout()
plt.show()
